# Etapa 4 — Validação Final do Agente AgTech

**Tema:** AgTech: Automação de Precisão com drones ou robôs agrícolas.

Este notebook executa o pipeline completo:

```text
Sensores -> RNA -> Sistema Especialista -> Gemini API -> Atuadores
```

A proposta usa uma **Rede Neural Artificial** para prever a umidade do solo na próxima hora. Essa previsão alimenta o **Sistema Especialista**, que decide irrigação, pulverização e inspeção por drone. Por fim, o resultado técnico é enviado ao **Gemini** para uma explicação em linguagem natural.

In [ ]:
# Instalação das bibliotecas necessárias para o Colab.
%pip install -q -U numpy pandas matplotlib scikit-learn google-genai

In [ ]:
# Importação das bibliotecas principais.
import json
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

BASE_DIR = Path('.')
DATA_DIR = BASE_DIR / 'data'
LOG_DIR = BASE_DIR / 'logs'
IMG_DIR = BASE_DIR / 'assets' / 'images'

for pasta in [DATA_DIR, LOG_DIR, IMG_DIR]:
    pasta.mkdir(parents=True, exist_ok=True)

FEATURES = [
    'temperatura_c', 'umidade_ar_pct', 'umidade_solo_pct', 'chuva_mm',
    'vento_kmh', 'ndvi', 'confianca_praga_pct', 'area_foliar_afetada_pct'
]
TARGET = 'umidade_solo_proxima_hora_pct'

## 1. Geração ou leitura dos sensores

Nesta versão acadêmica, os dados são simulados para representar sensores de campo. Em uma aplicação real, esta célula poderia ser substituída pela leitura de sensores IoT, telemetria de drone, estação climática ou banco de dados agrícola.

In [ ]:
def gerar_dados_sensores(n=720, seed=SEED):
    """Gera dados simulados de sensores agrícolas."""
    rng = np.random.default_rng(seed)
    horas = np.arange(n)
    timestamps = pd.date_range('2026-05-01 06:00:00', periods=n, freq='h')

    temperatura = 25 + 6*np.sin(2*np.pi*(horas % 24)/24) + rng.normal(0, 1.5, n)
    umidade_ar = np.clip(66 - 0.8*(temperatura - 25) + rng.normal(0, 5.0, n), 25, 95)
    chuva_mm = rng.gamma(shape=1.1, scale=2.4, size=n)
    chuva_mm[rng.random(n) < 0.65] = 0
    vento_kmh = np.clip(9 + rng.normal(0, 3, n), 1, 28)
    ndvi = np.clip(0.68 + rng.normal(0, 0.08, n), 0.35, 0.92)
    confianca_praga = np.clip(22 + 80*(0.72 - ndvi) + rng.normal(0, 15, n), 0, 100)
    area_foliar_afetada = np.clip(0.55*confianca_praga + rng.normal(0, 9, n), 0, 100)

    umidade_solo = np.clip(
        45 + 10*np.sin(2*np.pi*(horas % 168)/168)
        - 0.25*(temperatura - 25) + 0.35*chuva_mm
        - 0.06*area_foliar_afetada + rng.normal(0, 4.5, n),
        12, 82
    )

    umidade_futura = np.clip(
        umidade_solo - 0.55*np.maximum(temperatura - 24, 0)
        - 0.12*vento_kmh + 0.08*umidade_ar
        + 1.8*chuva_mm - 0.07*area_foliar_afetada
        + rng.normal(0, 2.0, n),
        8, 88
    )

    df = pd.DataFrame({
        'timestamp': timestamps,
        'temperatura_c': np.round(temperatura, 2),
        'umidade_ar_pct': np.round(umidade_ar, 2),
        'umidade_solo_pct': np.round(umidade_solo, 2),
        'chuva_mm': np.round(chuva_mm, 2),
        'vento_kmh': np.round(vento_kmh, 2),
        'ndvi': np.round(ndvi, 3),
        'confianca_praga_pct': np.round(confianca_praga, 2),
        'area_foliar_afetada_pct': np.round(area_foliar_afetada, 2),
        TARGET: np.round(umidade_futura, 2)
    })
    return df

try:
    df = gerar_dados_sensores()
    df.to_csv(DATA_DIR / 'sensores_agtech_simulados.csv', index=False)
    print('Base de sensores gerada com sucesso:', df.shape)
    display(df.head())
except Exception as erro:
    raise RuntimeError(f'Falha ao gerar/carregar sensores: {erro}')

## 2. Validação dos dados

Esta etapa evita que o pipeline quebre por ausência de colunas, valores nulos ou parâmetros inválidos.

In [ ]:
def validar_dataset(df):
    """Valida se o DataFrame possui as colunas e valores necessários."""
    colunas_obrigatorias = FEATURES + [TARGET]
    faltantes = sorted(set(colunas_obrigatorias) - set(df.columns))
    if faltantes:
        raise ValueError(f'Colunas ausentes no dataset: {faltantes}')
    if df[colunas_obrigatorias].isna().any().any():
        raise ValueError('Existem valores nulos nas colunas obrigatórias.')
    return True

def validar_leitura_sensor(leitura):
    """Valida uma leitura individual dos sensores."""
    faltantes = sorted(set(FEATURES) - set(leitura.keys()))
    if faltantes:
        raise ValueError(f'Leitura de sensores incompleta. Campos ausentes: {faltantes}')
    for chave in FEATURES:
        valor = leitura[chave]
        if valor is None or not np.isfinite(float(valor)):
            raise ValueError(f'Parâmetro inválido em {chave}: {valor}')
    return True

validar_dataset(df)
print('Dataset validado com sucesso.')

## 3. Treinamento da RNA

A RNA aprende a prever a umidade do solo na próxima hora. Essa previsão permite que o sistema antecipe a decisão de irrigação.

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, shuffle=False
)

modelo_rna = Pipeline([
    ('padronizador', StandardScaler()),
    ('rna', MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        random_state=SEED,
        max_iter=700,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30
    ))
])

modelo_rna.fit(X_train, y_train)
y_pred = modelo_rna.predict(X_test)
y_baseline = X_test['umidade_solo_pct'].to_numpy()

metricas = {
    'mae_rna': float(mean_absolute_error(y_test, y_pred)),
    'rmse_rna': float(mean_squared_error(y_test, y_pred) ** 0.5),
    'r2_rna': float(r2_score(y_test, y_pred)),
    'mae_baseline_sem_aprendizado': float(mean_absolute_error(y_test, y_baseline)),
    'rmse_baseline_sem_aprendizado': float(mean_squared_error(y_test, y_baseline) ** 0.5),
    'epocas_treinadas': int(modelo_rna.named_steps['rna'].n_iter_)
}

with open(LOG_DIR / 'metricas_modelo_final.json', 'w', encoding='utf-8') as arquivo:
    json.dump(metricas, arquivo, ensure_ascii=False, indent=2)

metricas

## 4. Gráficos de desempenho

A etapa final precisa evidenciar que o modelo aprendeu. Por isso, são gerados gráficos de Loss, previsão real vs. prevista e comparação antes/depois.

In [ ]:
loss_curve = modelo_rna.named_steps['rna'].loss_curve_

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(loss_curve)+1), loss_curve, marker='o', markersize=2)
plt.title('Evolução da Loss durante o treinamento da RNA')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(IMG_DIR / 'grafico_loss_rna.png', dpi=180)
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(y_test, y_pred, alpha=0.75)
min_val = min(float(np.min(y_test)), float(np.min(y_pred)))
max_val = max(float(np.max(y_test)), float(np.max(y_pred)))
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')
plt.title('Umidade real vs. umidade prevista pela RNA')
plt.xlabel('Umidade real próxima hora (%)')
plt.ylabel('Umidade prevista próxima hora (%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(IMG_DIR / 'previsao_vs_real_rna.png', dpi=180)
plt.show()

In [ ]:
n_show = min(80, len(y_test))
idx = np.arange(n_show)

plt.figure(figsize=(11, 5))
plt.plot(idx, y_test.iloc[:n_show].to_numpy(), label='Real próxima hora')
plt.plot(idx, y_baseline[:n_show], label='Antes: umidade atual usada como estimativa')
plt.plot(idx, y_pred[:n_show], label='Depois: previsão da RNA')
plt.title('Comparação antes e depois do aprendizado')
plt.xlabel('Amostras de teste')
plt.ylabel('Umidade do solo (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(IMG_DIR / 'comparacao_antes_depois.png', dpi=180)
plt.show()

## 5. Sistema Especialista

O Sistema Especialista representa a lógica da Etapa 2. Ele recebe a previsão da RNA e aplica regras SE/ENTÃO para definir os atuadores.

In [ ]:
def classificar_risco_praga(confianca_praga_pct, area_foliar_afetada_pct):
    """Classifica o risco de praga com base na visão computacional simulada."""
    if confianca_praga_pct >= 85 and area_foliar_afetada_pct >= 35:
        return 'crítico'
    if confianca_praga_pct >= 70 and area_foliar_afetada_pct >= 25:
        return 'alto'
    if confianca_praga_pct >= 50 or area_foliar_afetada_pct >= 18:
        return 'moderado'
    if confianca_praga_pct >= 30 or area_foliar_afetada_pct >= 10:
        return 'baixo'
    return 'normal'

def sistema_especialista(leitura, umidade_prevista_pct):
    """Aplica regras SE/ENTÃO para decidir irrigação, pulverização e inspeção."""
    validar_leitura_sensor(leitura)

    risco_praga = classificar_risco_praga(
        leitura['confianca_praga_pct'],
        leitura['area_foliar_afetada_pct']
    )

    chuva_detectada = leitura['chuva_mm'] >= 2.5
    vento_alto = leitura['vento_kmh'] >= 18

    if umidade_prevista_pct < 22 and not chuva_detectada:
        decisao_irrigacao, vazao = 'irrigação crítica', 100
    elif umidade_prevista_pct < 32 and not chuva_detectada:
        decisao_irrigacao, vazao = 'irrigação moderada', 65
    elif umidade_prevista_pct < 42:
        decisao_irrigacao, vazao = 'monitorar umidade', 25
    else:
        decisao_irrigacao, vazao = 'não irrigar', 0

    if risco_praga in ['alto', 'crítico'] and not vento_alto:
        manejo, pulverizador = 'acionar pulverização localizada e nova inspeção por drone', 'ligado'
    elif risco_praga in ['alto', 'crítico'] and vento_alto:
        manejo, pulverizador = 'adiar pulverização e priorizar nova inspeção por drone devido ao vento', 'bloqueado por vento'
    elif risco_praga == 'moderado':
        manejo, pulverizador = 'agendar nova varredura em até 24h', 'desligado'
    else:
        manejo, pulverizador = 'manter monitoramento padrão', 'desligado'

    return {
        'risco_praga': risco_praga,
        'decisao_irrigacao': decisao_irrigacao,
        'vazao_bomba_pct': vazao,
        'manejo_praga': manejo,
        'atuadores': {
            'bomba_irrigacao': 'ligada' if vazao > 0 else 'desligada',
            'vazao_bomba_pct': vazao,
            'pulverizador': pulverizador,
            'drone_inspecao': 'acionado' if risco_praga in ['moderado', 'alto', 'crítico'] else 'stand-by'
        }
    }

## 6. Integração com Gemini API

O Gemini não toma a decisão técnica. Ele apenas interpreta o resultado da RNA e do Sistema Especialista, explicando em linguagem natural o que aconteceu.

O código possui `try/except`. Caso a chave não esteja configurada ou a API falhe, o pipeline usa uma explicação local de contingência e não quebra.

In [ ]:
def explicar_com_gemini(resultado_pipeline):
    """Envia o resultado ao Gemini e usa fallback local em caso de erro."""
    prompt = f"""
Você é um especialista em AgTech, agricultura de precisão e automação agrícola.
Explique em linguagem natural a decisão abaixo, destacando:
1. O que a RNA aprendeu/previu.
2. Como o Sistema Especialista usou essa previsão.
3. Quais atuadores foram acionados.
4. Quais alertas estratégicos o produtor deve considerar.

Resultado técnico:
{json.dumps(resultado_pipeline, ensure_ascii=False, indent=2)}
"""
    try:
        api_key = os.getenv('GEMINI_API_KEY')

        # No Google Colab, a chave pode ser cadastrada em Secrets.
        try:
            from google.colab import userdata
            if not api_key:
                api_key = userdata.get('GEMINI_API_KEY')
        except Exception:
            pass

        if not api_key:
            raise RuntimeError('GEMINI_API_KEY não configurada.')

        from google import genai
        client = genai.Client(api_key=api_key)
        gemini_model = os.getenv('GEMINI_MODEL', 'gemini-2.0-flash-lite')
        response = client.models.generate_content(model=gemini_model, contents=prompt)
        return {'modo': 'gemini_api', 'texto': response.text}

    except Exception as erro:
        decisao = resultado_pipeline.get('decisao_especialista', {})
        texto_fallback = (
            'MODO CONTINGÊNCIA: A análise foi gerada localmente porque a API do Gemini '
            'não respondeu ou a chave não foi configurada. '
            f"A RNA previu umidade futura de {resultado_pipeline.get('umidade_prevista_proxima_hora_pct')}%. "
            f"O Sistema Especialista definiu '{decisao.get('decisao_irrigacao')}', "
            f"com risco de praga '{decisao.get('risco_praga')}'. "
            'Recomenda-se validar os sensores no campo e repetir a inspeção nas áreas críticas.'
        )
        return {'modo': 'fallback_local', 'erro': str(erro), 'texto': texto_fallback}

## 7. Pipeline final integrado

Esta célula executa o fluxo completo sem intervenção manual:

```text
Sensores -> RNA -> Sistema Especialista -> Gemini/Fallback -> Atuadores
```

In [ ]:
def executar_pipeline(leitura):
    """Executa o pipeline final do agente AgTech."""
    validar_leitura_sensor(leitura)

    entrada_modelo = pd.DataFrame([leitura])[FEATURES]
    umidade_prevista = float(modelo_rna.predict(entrada_modelo)[0])

    decisao = sistema_especialista(leitura, umidade_prevista)

    resultado = {
        'leitura_sensores': leitura,
        'umidade_prevista_proxima_hora_pct': round(umidade_prevista, 2),
        'decisao_especialista': decisao
    }

    resultado['analise_interpretativa'] = explicar_com_gemini(resultado)
    return resultado

try:
    leitura_atual = df.iloc[-1][FEATURES].astype(float).to_dict()
    resultado_final = executar_pipeline(leitura_atual)

    with open(LOG_DIR / 'log_pipeline_final.json', 'w', encoding='utf-8') as arquivo:
        json.dump(resultado_final, arquivo, ensure_ascii=False, indent=2)

    print(json.dumps(resultado_final, ensure_ascii=False, indent=2))
except Exception as erro:
    print(f'Erro controlado no pipeline final: {erro}')

## 8. Imagem da arquitetura e log visual

As imagens salvas nesta etapa podem ser usadas diretamente no README do GitHub.

In [ ]:
plt.figure(figsize=(12, 4.8))
ax = plt.gca()
ax.set_axis_off()

caixas = [
    ('Sensores\nsolo, clima, NDVI, pragas', 0.05, 0.55),
    ('RNA\nprevê umidade futura', 0.26, 0.55),
    ('Sistema Especialista\nregras SE/ENTÃO', 0.47, 0.55),
    ('Gemini API\nanálise interpretativa', 0.68, 0.55),
    ('Atuadores\nbomba, pulverizador, drone', 0.47, 0.15),
]

for texto, x, y0 in caixas:
    ax.text(x, y0, texto, ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='black'))

setas = dict(arrowstyle='->', lw=1.5)
ax.annotate('', xy=(0.20, 0.55), xytext=(0.14, 0.55), arrowprops=setas)
ax.annotate('', xy=(0.41, 0.55), xytext=(0.32, 0.55), arrowprops=setas)
ax.annotate('', xy=(0.62, 0.55), xytext=(0.53, 0.55), arrowprops=setas)
ax.annotate('', xy=(0.47, 0.26), xytext=(0.47, 0.45), arrowprops=setas)
ax.text(0.5, 0.92, 'Arquitetura final do agente AgTech', ha='center', fontsize=14, weight='bold')

plt.tight_layout()
plt.savefig(IMG_DIR / 'arquitetura_pipeline_agtech.png', dpi=180)
plt.show()

In [ ]:
log_text = json.dumps(resultado_final, ensure_ascii=False, indent=2)
    log_preview = '
'.join(log_text.splitlines()[:32])

    plt.figure(figsize=(12, 8))
    ax = plt.gca()
    ax.set_axis_off()
    ax.text(0.01, 0.99, 'Log final de execução do pipeline

' + log_preview,
            va='top', ha='left', fontsize=9, family='monospace')
    plt.tight_layout()
    plt.savefig(IMG_DIR / 'log_execucao_pipeline.png', dpi=180)
    plt.show()

## 9. Conclusão

O pipeline final integra todas as etapas do projeto:

- dados de sensores definidos na Etapa 1;
- Sistema Especialista da Etapa 2;
- RNA preditiva da Etapa 3;
- validação final, tratamento de exceções, gráficos, logs e documentação visual da Etapa 4.

O sistema está pronto para ser colocado em um repositório GitHub público, com pastas organizadas e sem arquivos compactados.